[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/67_gpu_batch_scheduling_solution.ipynb)

# Solution: GPU Batch Scheduling (Min Batches)

Reference solution — sort + two pointers + DP with diagonal prefix-min.


In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
from typing import List


In [ ]:
# ✅ SOLUTION

class Solution:
    def min_batches(self, lengths: List[int], m: int, k: int, c: int) -> int:
        n = len(lengths)
        m = min(m, n)
        a = [0] + sorted(lengths)          # 1-based
        INF = float('inf')
        # f[j][i]: min batches using first i sorted requests, dropping exactly j
        f = [[INF] * (n + 1) for _ in range(m + 1)]
        # g[j][i]: prefix-min of f along the diagonal (p - q constant)
        g = [[INF] * (n + 1) for _ in range(m + 1)]
        f[0][0] = 0
        g[0][0] = 0
        left = 1
        for i in range(1, n + 1):
            while a[i] - a[left] > k:      # smallest left with a[i]-a[left] <= k
                left += 1
            s = max(left - 1, i - c)       # requests before s must be dropped for this batch
            for j in range(m + 1):
                best = INF
                if j > 0:                  # drop request i
                    best = f[j - 1][i - 1]
                p = max(s, j)              # keep i, batch needs no extra drop
                if p < i:
                    best = min(best, f[j][p] + 1)
                if j > 0 and s > 0:        # keep i, batch needs extra drops (diagonal min)
                    best = min(best, g[j - 1][s - 1] + 1)
                f[j][i] = best
                g[j][i] = best
                if j > 0:
                    g[j][i] = min(g[j][i], g[j - 1][i - 1])
        return min(f[j][n] for j in range(m + 1))


In [ ]:
# Demo
sol = Solution()
print(sol.min_batches([10, 11, 15, 16, 17], 1, 1, 2))  # 2
print(sol.min_batches([1, 100, 2, 200], 2, 10, 3))     # 1


In [ ]:
from torch_judge import check
check('gpu_batch_scheduling')
